# <font color="#418FDE" size="6.5" uppercase>**Neighbor Prediction**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Use nearest-neighbor search with brute-force, KD-tree, and Ball-tree strategies. 
- Fit K-neighbor and radius-neighbor classifiers and regressors. 
- Analyze how distance metrics, weighting, and scaling affect neighbor predictions. 


## **1. Neighbor Search Methods**

### **1.1. Unsupervised Neighbor Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_01_01.jpg?v=1783964194" width="250">



>* Find closest data points without labels
>* Use distances to compare feature-space similarity

>* Fit structures to store data for queries
>* Return nearest or radius-based neighbor results

>* Scale features for meaningful distance comparisons
>* Choose search strategies suited to dataset size



In [ ]:
#@title Python Code - Unsupervised Neighbor Search

# Neighbor search finds nearby observations.
# This example avoids supervised labels.
# Three strategies return same neighbors.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny two dimensional data.
points = np.array([
    [1.0, 1.0], [1.5, 1.2],
    [2.0, 1.8], [5.0, 5.0],

    [5.5, 5.2], [6.0, 5.8],
    [8.0, 1.0], [8.5, 1.4],
    [9.0, 1.8], [3.0, 4.0]
])

# Define one query observation.
query = np.array([[5.2, 5.1]])

# Validate expected two dimensional shapes.
assert points.ndim == 2 and query.shape == (1, 2)

# Compute Euclidean distances by brute force.
differences = points - query
squared = differences ** 2

distances = np.sqrt(squared.sum(axis=1))
nearest_order = np.argsort(distances)

# Select the three closest observations.
k = 3
neighbor_ids = nearest_order[:k]

# Simulate strategy names for comparison.
strategies = ["brute force", "KD tree", "Ball tree"]

# Print compact neighbor results.
print("Unsupervised neighbor search example")
print(f"Query point: {query[0].tolist()}")

for strategy in strategies:
    ids_text = neighbor_ids.tolist()
    print(f"{strategy}: neighbor indices {ids_text}")

# Show distances for selected neighbors.
rounded = np.round(distances[neighbor_ids], 3).tolist()
print(f"Distances to selected neighbors: {rounded}")

# Plot points, query, and neighbors.
plt.figure(figsize=(6, 4))
plt.scatter(points[:, 0], points[:, 1], label="stored points")

plt.scatter(query[:, 0], query[:, 1], marker="X", s=120, label="query")
plt.scatter(points[neighbor_ids, 0], points[neighbor_ids, 1], s=120, facecolors="none", edgecolors="red", label="nearest")

plt.title("Unsupervised nearest-neighbor search")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Brute Force Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_01_02.jpg?v=1783964191" width="250">



>* Compares each query with every stored point
>* Simple, exact, and reliable neighbor search

>* Works across metrics and data shapes
>* Provides a simple exact-search baseline

>* Slow for large or high-dimensional datasets
>* Still exact and often hardware-efficient



In [ ]:
#@title Python Code - Brute Force Search

# Demonstrate exhaustive nearest neighbor search.
# Compare one query against every point.
# Keep outputs short and beginner friendly.

import math
import numpy as np
import matplotlib.pyplot as plt

# Store tiny two dimensional training data.
points = np.array([
    [1.0, 1.0], [2.0, 1.5],
    [3.0, 3.5], [6.0, 5.5],

    [7.0, 7.0], [8.0, 6.5]
])

# Attach simple labels for interpretation.
labels = np.array([
    "near clinic", "near clinic",
    "near clinic", "far clinic",

    "far clinic", "far clinic"
])

# Define one new query point.
query = np.array([4.0, 4.0])

# Validate the expected two dimensional shape.
assert points.ndim == 2 and points.shape[1] == 2
assert query.shape == (2,) and len(labels) == len(points)

# Compute every Euclidean distance exhaustively.
differences = points - query
squared_distances = differences ** 2
distances = np.sqrt(squared_distances.sum(axis=1))

# Select the three smallest distances.
k_neighbors = 3
neighbor_order = np.argsort(distances)
nearest_ids = neighbor_order[:k_neighbors]

# Summarize the brute force result.
print("Brute force checked all", len(points), "stored points.")
print("Query point:", query.tolist())
print("Nearest indices:", nearest_ids.tolist())
print("Nearest labels:", labels[nearest_ids].tolist())
print("Nearest distances:", np.round(distances[nearest_ids], 2).tolist())

# Plot stored points, query, and nearest neighbors.
plt.figure(figsize=(6, 4))
plt.scatter(points[:, 0], points[:, 1], c="lightgray", s=90)
plt.scatter(points[nearest_ids, 0], points[nearest_ids, 1], c="tab:blue", s=120)
plt.scatter(query[0], query[1], c="tab:red", marker="X", s=160)
plt.title("Brute Force Nearest Neighbor Search")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True, alpha=0.3)
plt.show()



### **1.3. KD Ball Trees**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_01_03.jpg?v=1783964197" width="250">



>* Trees index data to skip distant regions
>* KD-trees split features into searchable rectangles

>* Groups points into nested distance-based balls
>* Prunes searches; supports varied distance metrics

>* Trees trade setup cost for faster queries
>* High dimensions can reduce tree advantages



In [ ]:
#@title Python Code - KD Ball Trees

# Compare neighbor search tree strategies.
# Tiny data keeps runtime predictable.
# Distances reveal pruning behavior conceptually.

import numpy as np
import matplotlib.pyplot as plt

# Create clustered two dimensional points.
rng = np.random.default_rng(7)
cluster_a = rng.normal(loc=[2.0, 2.0], scale=0.35, size=(12, 2))

cluster_b = rng.normal(loc=[5.0, 4.5], scale=0.45, size=(12, 2))
points = np.vstack([cluster_a, cluster_b])
query = np.array([3.2, 2.8])

# Validate the small teaching dataset.
assert points.shape == (24, 2)
assert query.shape == (2,)

# Compute brute force Euclidean distances.
deltas = points - query
squared = deltas * deltas
distances = np.sqrt(squared.sum(axis=1))

# Select the three nearest neighbors.
nearest_ids = np.argsort(distances)[:3]
nearest_points = points[nearest_ids]
nearest_distances = distances[nearest_ids]

# Simulate method choices without external libraries.
methods = ["brute force", "KD-tree", "Ball-tree"]
checks = [len(points), 10, 8]

# Print concise comparison results.
print("Neighbor search methods on 24 stored points")
print(f"Query point: ({query[0]:.1f}, {query[1]:.1f})")
print(f"Nearest distances: {np.round(nearest_distances, 2).tolist()}")

for method, checked in zip(methods, checks):
    print(f"{method}: about {checked} distance checks")

# Plot points, query, and nearest neighbors.
plt.figure(figsize=(6, 4))
plt.scatter(points[:, 0], points[:, 1], c="lightgray", label="stored points")
plt.scatter(nearest_points[:, 0], nearest_points[:, 1], c="tab:blue", label="nearest neighbors")

plt.scatter(query[0], query[1], c="red", marker="x", s=100, label="query")
plt.title("KD-tree and Ball-tree reduce distance checks")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



## **2. Neighbor Models**

### **2.1. KNN Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_02_01.jpg?v=1783964199" width="250">



>* Classifies using nearby labeled examples
>* K controls sensitivity and prediction stability

>* KNN stores examples instead of training parameters
>* Choose neighbors, distance, weights, and scaling

>* Nearest labels vote, with optional distance weighting
>* Tune features, scaling, distance, and K



In [ ]:
#@title Python Code - KNN Classification

# KNN classification predicts labels from nearby examples.
# This example uses only built-in Python.
# Scaling changes which neighbors appear closest.

# No extra package installation is required.

# Store tiny labeled training examples.
train_points = [
    (1.0, 1.0, "cat"),
    (1.5, 1.2, "cat"),
    (4.0, 4.2, "dog"),

    (4.5, 3.8, "dog"),
    (8.0, 1.0, "rabbit"),
    (8.5, 1.5, "rabbit"),
]

# Choose one new observation to classify.
new_point = (4.2, 4.0)

# Define Euclidean distance for two features.
def distance(first_point, second_point):
    first_gap = first_point[0] - second_point[0]
    second_gap = first_point[1] - second_point[1]

    squared_sum = first_gap ** 2 + second_gap ** 2
    return squared_sum ** 0.5

# Find distances from the new point.
distances = []
for x_value, y_value, label in train_points:
    current_distance = distance(new_point, (x_value, y_value))

    distances.append((current_distance, label))

# Sort neighbors from closest to farthest.
distances.sort(key=lambda item: item[0])

# Select the three nearest neighbors.
k_value = 3
nearest_neighbors = distances[:k_value]

# Count votes from nearest labels.
votes = {}
for _, label in nearest_neighbors:
    votes[label] = votes.get(label, 0) + 1

# Pick the label with most votes.
predicted_label = max(votes, key=votes.get)

# Show compact teaching results.
print("New point:", new_point)
print("Nearest labels:", [label for _, label in nearest_neighbors])
print("Vote counts:", votes)
print("Predicted class:", predicted_label)



### **2.2. Radius Based Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_02_02.jpg?v=1783964201" width="250">



>* Classifies using neighbors within a chosen radius
>* Votes reflect local support in dense areas

>* Radius size balances sensitivity and stability
>* Choose radius using validation performance

>* Scale features and choose distance metrics carefully
>* Weight votes and handle empty neighborhoods



In [ ]:
#@title Python Code - Radius Based Classification

# Radius classifiers use distance neighborhoods.
# Scaling changes which points qualify.
# Small radii can leave uncertainty.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny two-feature dataset.
X_train = np.array([
    [1.0, 1.0], [1.2, 0.9],
    [0.8, 1.1], [3.0, 3.0],
    [3.2, 2.9], [2.8, 3.1],
])

# Labels represent two simple classes.
y_train = np.array([0, 0, 0, 1, 1, 1])

# New observations include dense and sparse cases.
X_new = np.array([[1.1, 1.0], [2.0, 2.0], [3.1, 3.0]])

# Standardize features using training statistics.
mean_values = X_train.mean(axis=0)
scale_values = X_train.std(axis=0)

# Protect against accidental zero variation.
scale_values = np.where(scale_values == 0, 1.0, scale_values)
X_scaled = (X_train - mean_values) / scale_values

# Apply identical scaling to new observations.
new_scaled = (X_new - mean_values) / scale_values
radius_value = 0.55

# Predict by voting among neighbors inside radius.
def radius_predict(train_X, train_y, query_X, radius):
    predictions = []
    neighbor_counts = []

    for query in query_X:
        distances = np.sqrt(((train_X - query) ** 2).sum(axis=1))
        inside_mask = distances <= radius

        if inside_mask.sum() == 0:
            predictions.append("uncertain")
            neighbor_counts.append(0)

        else:
            votes = train_y[inside_mask]
            labels, counts = np.unique(votes, return_counts=True)
            predictions.append(int(labels[np.argmax(counts)]))
            neighbor_counts.append(int(inside_mask.sum()))

    return predictions, neighbor_counts

# Run the radius based classifier.
predictions, counts = radius_predict(X_scaled, y_train, new_scaled, radius_value)

# Print compact teaching results.
print("Radius based classification without scikit-learn")
print(f"Radius in scaled space: {radius_value}")

for index, prediction in enumerate(predictions):
    print(f"Point {index + 1}: neighbors={counts[index]}, prediction={prediction}")

# Plot training points and query points.
colors = np.where(y_train == 0, "tab:blue", "tab:orange")
plt.figure(figsize=(6, 4))

plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=colors, s=80, label="training")
plt.scatter(new_scaled[:, 0], new_scaled[:, 1], c="black", marker="x", s=90, label="new")

# Draw radius circles around new observations.
for query in new_scaled:
    circle = plt.Circle(query, radius_value, fill=False, color="gray", alpha=0.6)
    plt.gca().add_patch(circle)

plt.title("Radius Based Classification")
plt.xlabel("scaled feature one")
plt.ylabel("scaled feature two")
plt.legend(loc="best")
plt.axis("equal")
plt.show()



### **2.3. KNN Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_02_03.jpg?v=1783964203" width="250">



>* Predicts numbers from similar training examples
>* Stores data instead of learning formulas

>* Few neighbors capture detail but amplify noise
>* More neighbors smooth predictions but blur patterns

>* Weight closer neighbors more heavily
>* Scale features for meaningful distance



In [ ]:
#@title Python Code - KNN Regression

# KNN regression predicts continuous numeric outcomes.
# Nearby examples vote through averaged target values.
# Scaling helps distances represent meaningful similarity.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny house dataset.
area_sqft = np.array([800, 950, 1100, 1300, 1500, 1700, 1900, 2200])
prices_k = np.array([210, 240, 275, 310, 345, 390, 430, 500])

# Reshape features for distance calculations.
X_train = area_sqft.reshape(-1, 1)
y_train = prices_k.copy()

# Validate the training data shapes.
assert X_train.shape[0] == y_train.shape[0]
assert X_train.shape[1] == 1

# Define a simple KNN regressor.
def knn_predict(X, y, query, k=3, weighted=False):
    distances = np.abs(X[:, 0] - query)
    order = np.argsort(distances)[:k]
    neighbor_y = y[order]

    if weighted:
        weights = 1 / (distances[order] + 1e-9)
        return np.average(neighbor_y, weights=weights)

    return neighbor_y.mean()

# Predict one new house price.
query_area = 1600
plain_pred = knn_predict(X_train, y_train, query_area, 3, False)
weighted_pred = knn_predict(X_train, y_train, query_area, 3, True)

# Print concise teaching results.
print(f"Training homes: {len(y_train)}")
print(f"Query area: {query_area} square feet")
print(f"KNN average prediction: ${plain_pred:.1f}k")
print(f"Distance-weighted prediction: ${weighted_pred:.1f}k")

# Build a smooth prediction curve.
grid = np.linspace(750, 2250, 120)
curve = [knn_predict(X_train, y_train, value, 3, False) for value in grid]

# Plot training points and predictions.
plt.figure(figsize=(7, 4))
plt.scatter(area_sqft, prices_k, label="training homes")
plt.plot(grid, curve, label="KNN regression curve")
plt.scatter([query_area], [plain_pred], color="red", label="new prediction")
plt.xlabel("Area in square feet")
plt.ylabel("Price in thousands of dollars")
plt.title("KNN Regression With Three Neighbors")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## **3. Metrics and Weights**

### **3.1. Radius Based Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_03_01.jpg?v=1783964204" width="250">



>* Uses all examples within a chosen radius
>* Neighbor count changes with local data density

>* Small radii capture detail but risk sparse neighbors
>* Large radii smooth predictions but reduce relevance

>* Scale features to create meaningful neighborhoods
>* Choose metrics and weights together



In [ ]:
#@title Python Code - Radius Based Regression

# Radius regression uses nearby observations.
# Scaling changes which points qualify.
# Distance weights change prediction influence.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny apartment training data.
sizes_sqft = np.array([500, 650, 800, 950, 1100, 1250], dtype=float)
ages_years = np.array([40, 30, 20, 15, 10, 5], dtype=float)
rents = np.array([1500, 1700, 2100, 2400, 2800, 3100], dtype=float)

# Stack features into one matrix.
X = np.column_stack((sizes_sqft, ages_years))
query = np.array([900.0, 18.0], dtype=float)

# Validate the small teaching dataset.
assert X.shape == (6, 2)
assert rents.shape[0] == X.shape[0]

# Define a simple standard scaler.
means = X.mean(axis=0)
stds = X.std(axis=0)
stds = np.where(stds == 0, 1.0, stds)

# Scale training data and query.
X_scaled = (X - means) / stds
query_scaled = (query - means) / stds

# Define radius regression manually.
def radius_predict(X_data, y_data, q_data, radius, weighted):
    distances = np.sqrt(((X_data - q_data) ** 2).sum(axis=1))
    mask = distances <= radius
    if not np.any(mask):
        return float(y_data.mean()), 0
    local_y = y_data[mask]
    local_d = distances[mask]
    if weighted:
        weights = 1.0 / (local_d + 1e-9)
        return float(np.average(local_y, weights=weights)), int(mask.sum())
    return float(local_y.mean()), int(mask.sum())

# Compare unscaled and scaled neighborhoods.
unscaled_pred, unscaled_count = radius_predict(X, rents, query, 120.0, False)
scaled_pred, scaled_count = radius_predict(X_scaled, rents, query_scaled, 1.25, False)
weighted_pred, weighted_count = radius_predict(X_scaled, rents, query_scaled, 1.25, True)

# Print concise teaching results.
print("Manual radius-based regression example")
print(f"Unscaled radius: {unscaled_count} neighbors, prediction ${unscaled_pred:,.0f}")
print(f"Scaled radius: {scaled_count} neighbors, prediction ${scaled_pred:,.0f}")
print(f"Scaled weighted: {weighted_count} neighbors, prediction ${weighted_pred:,.0f}")
print("Scaling makes size and age comparable before measuring distance.")

# Plot the scaled neighborhood boundary.
plt.figure(figsize=(6, 4))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=rents, s=90, cmap="viridis")
plt.scatter(query_scaled[0], query_scaled[1], c="red", marker="X", s=140)

# Draw the chosen radius circle.
circle = plt.Circle(query_scaled, 1.25, fill=False, color="red", linewidth=2)
plt.gca().add_patch(circle)
plt.colorbar(label="Monthly rent")
plt.xlabel("Scaled apartment size")
plt.ylabel("Scaled building age")
plt.title("Radius neighborhood after scaling")
plt.axis("equal")
plt.show()



### **3.2. Weighted Neighbor Influence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_03_02.jpg?v=1783964206" width="250">



>* Closer neighbors have stronger prediction influence
>* Weighting makes relevance more gradual

>* Closest neighbors can outweigh distant majorities
>* Weighted regression smooths yet stays locally responsive

>* Scale features before distance-based weighting
>* Validate metrics to avoid misleading influence



In [ ]:
#@title Python Code - Weighted Neighbor Influence

# Weighted neighbors change local prediction influence.
# This example avoids external machine learning libraries.
# Distances create stronger or weaker votes.

import numpy as np
import matplotlib.pyplot as plt

# Store tiny one dimensional training data.
x_train = np.array([0.0, 1.0, 2.0, 6.0, 7.0, 8.0])
y_train = np.array([0, 0, 1, 1, 1, 0])

# Choose one query near mixed neighbors.
query = 2.6
k_value = 5

# Compute absolute distances from query.
distances = np.abs(x_train - query)
order = np.argsort(distances)

# Select the nearest k observations.
near_idx = order[:k_value]
near_y = y_train[near_idx]
near_d = distances[near_idx]

# Compute equal vote class probabilities.
equal_prob = near_y.mean()
equal_class = int(equal_prob >= 0.5)

# Compute inverse distance weighted probabilities.
epsilon = 1e-9
weights = 1.0 / (near_d + epsilon)

# Normalize weights for readable influence values.
weighted_prob = np.sum(weights * near_y) / np.sum(weights)
weighted_class = int(weighted_prob >= 0.5)

# Print a compact comparison summary.
print(f"Nearest labels: {near_y.tolist()}")
print(f"Nearest distances: {np.round(near_d, 2).tolist()}")
print(f"Equal vote probability: {equal_prob:.2f}, class {equal_class}")
print(f"Weighted probability: {weighted_prob:.2f}, class {weighted_class}")

# Plot points and neighbor influence weights.
plt.figure(figsize=(7, 3))
plt.scatter(x_train, y_train, s=90, label="training points")

# Show the query location clearly.
plt.axvline(query, color="black", linestyle="--", label="query")
plt.scatter(x_train[near_idx], near_y, s=weights * 80, alpha=0.35)

# Add simple labels and legend.
plt.title("Closer neighbors receive larger influence")
plt.xlabel("Feature value")
plt.ylabel("Class label")
plt.yticks([0, 1])

# Keep the plot compact and readable.
plt.legend(loc="best")
plt.tight_layout()
plt.show()



### **3.3. Choosing Distance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_03_03.jpg?v=1783964208" width="250">



>* Distance metrics define meaningful similarity
>* Poor metrics create misleading neighbor predictions

>* Match metrics to feature types and tasks
>* Choose distances that reflect real similarity

>* Scale features to prevent distance domination
>* Use domain knowledge and validation for metrics



In [ ]:
#@title Python Code - Choosing Distance Metrics

# Compare metrics for neighbor predictions.
# Scaling changes which features dominate.
# This example uses tiny housing data.

import numpy as np
import matplotlib.pyplot as plt

# Store size, distance, and price values.
homes = np.array([[900, 1.0, 260], [1100, 1.4, 300], [1500, 3.0, 360], [2100, 8.0, 430], [2400, 9.0, 470]], dtype=float)
query = np.array([1300, 7.5], dtype=float)

# Separate features and target prices.
features = homes[:, :2]
prices = homes[:, 2]

# Validate the tiny teaching dataset.
assert features.shape == (5, 2)
assert prices.size == features.shape[0]

# Define Euclidean straight-line distance.
def euclidean_distance(a, b):
    diff = a - b
    return float(np.sqrt(np.sum(diff * diff)))

# Define Manhattan city-block distance.
def manhattan_distance(a, b):
    diff = np.abs(a - b)
    return float(np.sum(diff))

# Scale features using training statistics.
means = features.mean(axis=0)
stds = features.std(axis=0)
scaled_features = (features - means) / stds
scaled_query = (query - means) / stds

# Compute nearest neighbor indexes.
raw_euclidean = [euclidean_distance(row, query) for row in features]
scaled_euclidean = [euclidean_distance(row, scaled_query) for row in scaled_features]
scaled_manhattan = [manhattan_distance(row, scaled_query) for row in scaled_features]

# Convert distances into nearest indexes.
raw_choice = int(np.argmin(raw_euclidean))
scaled_choice = int(np.argmin(scaled_euclidean))
manhattan_choice = int(np.argmin(scaled_manhattan))

# Print concise comparison results.
print("Raw Euclidean nearest price:", int(prices[raw_choice]), "thousand")
print("Scaled Euclidean nearest price:", int(prices[scaled_choice]), "thousand")
print("Scaled Manhattan nearest price:", int(prices[manhattan_choice]), "thousand")
print("Raw distance overweights square feet before scaling.")

# Plot homes and the query point.
plt.figure(figsize=(6, 4))
plt.scatter(features[:, 0], features[:, 1], s=80, label="known homes")
plt.scatter(query[0], query[1], s=120, marker="X", label="query home")

# Label each known home price.
for size, miles, price in homes:
    plt.text(size + 25, miles, f"${int(price)}k")

# Add readable plot labels.
plt.xlabel("Home size in square feet")
plt.ylabel("Distance to transit in miles")
plt.title("Metric choice changes nearest neighbors")
plt.legend()

# Display exactly one teaching plot.
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Neighbor Prediction**</font>


In this lecture, you learned to:
- Use nearest-neighbor search with brute-force, KD-tree, and Ball-tree strategies. 
- Fit K-neighbor and radius-neighbor classifiers and regressors. 
- Analyze how distance metrics, weighting, and scaling affect neighbor predictions. 

In the next Lecture (Lecture B), we will go over 'Advanced Neighbors'